In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

###  customers_dataset

- customer_id: Each order has a unique customer_id.
- customer_unique_id: unique identifier of a customer.
- customer_zip_code_prefix: first five digits of customer zip code


In [6]:
customers = pd.read_csv(r"data\original_data\olist_customers_dataset.csv", dtype={'customer_zip_code_prefix':str})
customers.head()


## We are using str datatype explicitly for zip_code in - customers, geolocation, sellers 
## This is because pandas read that as integer - which caused some issues

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## geolocation_dataset

#### Remove Duplicates from "data\original_data\olist_geolocation_dataset.csv" 
- This csv contains duplicated values in primary index - geolocation_zip_code_prefix
- Take mean of lat, lng
- Take first instance of city and state

In [7]:
geolocation_df = pd.read_csv(r"data\original_data\olist_geolocation_dataset.csv", dtype={'geolocation_zip_code_prefix': str})
print("Number of Duplicates: ", geolocation_df.duplicated().sum())
print("Number of Duplicated index: ", geolocation_df['geolocation_zip_code_prefix'].duplicated().sum())
print("Check the duplicates-")
geolocation_df[geolocation_df['geolocation_zip_code_prefix']=="01046"].head()

Number of Duplicates:  261831
Number of Duplicated index:  981148
Check the duplicates-


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
15,01046,-23.546081,-46.644820,sao paulo,SP
19,01046,-23.545884,-46.643163,sao paulo,SP
20,01046,-23.545166,-46.643786,sao paulo,SP


In [8]:
geolocation_cleaned = geolocation_df.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat':'mean',
    'geolocation_lng':'mean',
    'geolocation_city': 'first',
    'geolocation_state':'first'
    }).reset_index()

print("Number of Duplicates: ", geolocation_cleaned.duplicated().sum())
print("Number of Duplicated index: ", geolocation_cleaned['geolocation_zip_code_prefix'].duplicated().sum())
geolocation_cleaned.head()

Number of Duplicates:  0
Number of Duplicated index:  0


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01001,-23.550190,-46.634024,sao paulo,SP
1,01002,-23.548146,-46.634979,sao paulo,SP
2,01003,-23.548994,-46.635731,sao paulo,SP
3,01004,-23.549799,-46.634757,sao paulo,SP
4,01005,-23.549456,-46.636733,sao paulo,SP


In [9]:
print("Original Length of dataset: ", len(geolocation_df))
print("Lenth of cleaned dataset: ", len(geolocation_cleaned))

Original Length of dataset:  1000163
Lenth of cleaned dataset:  19015


- ___Foreign key constraint issue in `sellers` and `customers`___   


  
Check if all the zip codes in `customers`and `sellers` are in `geolocation`
Since we will use zip_codes as foreign key 

In [42]:
sellers = pd.read_csv(r'data\original_data\olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix':str})

In [43]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [47]:
customers_zip_codes = set(customers['customer_zip_code_prefix'])
sellers_zip_codes = set(sellers['seller_zip_code_prefix'])
geolocation_zip_codes = set(geolocation_cleaned['geolocation_zip_code_prefix'])

needed_codes = customers_zip_codes | sellers_zip_codes
missing_zip_codes = sorted(list(needed_codes - geolocation_zip_codes))

print("Needed - Geolocation (missing number of codes): ", len(missing_zip_codes))
print("Geolocation - Needed: ", len(geolocation_zip_codes - needed_codes))

print("Zip Codes missing from geolocation:\n", missing_zip_codes)

Needed - Geolocation (missing number of codes):  162
Geolocation - Needed:  4099
Zip Codes missing from geolocation:
 ['02140', '02285', '06930', '07412', '07430', '07729', '07784', '08342', '08980', '11547', '12332', '12770', '13307', '17390', '19740', '25840', '25919', '27980', '28120', '28160', '28388', '28530', '28575', '28617', '28655', '29196', '29718', '29949', '35104', '35242', '35408', '36248', '36596', '36857', '36956', '37005', '37708', '38627', '38710', '39103', '41098', '41347', '42716', '42843', '43870', '44135', '48504', '49870', '55027', '55863', '56327', '56485', '57254', '58286', '58734', '59299', '59547', '61906', '62625', '62898', '64047', '64095', '64605', '65137', '65830', '67105', '68511', '68629', '70316', '70324', '70333', '70686', '70701', '70702', '70716', '71208', '71261', '71539', '71551', '71574', '71590', '71591', '71593', '71676', '71698', '71810', '71884', '71905', '71919', '71953', '71971', '71975', '71976', '71993', '71995', '71996', '72002', '72005',

- We need all the zip codes in customers and sellers to be in geolocation 
    - Because we will use the geolocation_zip_code_prefix as foreign key in sellers, and customers table 

We have two options -     
A. Drop Missing Zip Codes  
B. Add Zip Codes in geolocation with all other fields as Null (or find out those values (i aint doing ts))
    

In [48]:
## Add Zip Codes with other fields as null
missing_zip_codes_df = pd.DataFrame(
    {
        'geolocation_zip_code_prefix': missing_zip_codes,
        'geolocation_lat': np.nan,
        'geolocation_lng': np.nan,
        'geolocation_city': np.nan,
        'geolocation_state': np.nan
    }).astype(
        {'geolocation_zip_code_prefix': str, 'geolocation_lat':np.float64, 'geolocation_lng': np.float64, 'geolocation_city': str, 'geolocation_state':str}
    )
print(missing_zip_codes_df.shape)
missing_zip_codes_df.head()

(162, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,02140,NaN,NaN,NaN,NaN
1,02285,NaN,NaN,NaN,NaN
2,06930,NaN,NaN,NaN,NaN
3,07412,NaN,NaN,NaN,NaN
4,07430,NaN,NaN,NaN,NaN


In [49]:
missing_zip_codes_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 162 entries, 0 to 161
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   geolocation_zip_code_prefix  162 non-null    str    
 1   geolocation_lat              0 non-null      float64
 2   geolocation_lng              0 non-null      float64
 3   geolocation_city             0 non-null      str    
 4   geolocation_state            0 non-null      str    
dtypes: float64(2), str(3)
memory usage: 6.5 KB


In [50]:
print(geolocation_cleaned.shape, missing_zip_codes_df.shape)
geolocation_cleaned_final = pd.concat([geolocation_cleaned, missing_zip_codes_df], axis=0, join='outer')
print(geolocation_cleaned_final.shape)

(19015, 5) (162, 5)
(19177, 5)


In [ ]:
## Save the cleaned geolocation data
geolocation_cleaned_final.to_csv('data/transformed_data/olist_geolocation_dataset_cleaned.csv',index=False)

In [57]:
final_check = pd.read_csv('data/transformed_data/olist_geolocation_dataset_cleaned.csv', dtype={'geolocation_zip_code_prefix': str})

customers_zip_codes = (
    customers['customer_zip_code_prefix']
)

sellers_zip_codes = (
    sellers['seller_zip_code_prefix']
)

geolocation_zip_codes = (
    final_check['geolocation_zip_code_prefix']
)

missing_zip_codes = (set(customers_zip_codes) | set(sellers_zip_codes)) - set(geolocation_zip_codes)

len(missing_zip_codes), sorted(missing_zip_codes)

(0, [])

## order_items_dataset
- order_id and order_item_id together make primary key

In [31]:
order_items = pd.read_csv(r'data\original_data\olist_order_items_dataset.csv')
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [39]:
print("Duplicates in 'order_id':",order_items['order_id'].duplicated().sum())

print("Duplicates in 'order_item_id':",order_items['order_item_id'].duplicated().sum())
print("Unique order items id:", order_items['order_item_id'].unique())

print("Duplicates in 'product_id':",order_items['product_id'].duplicated().sum())

Duplicates in 'order_id': 13984
Duplicates in 'order_item_id': 112629
Unique order items id: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21]
Duplicates in 'product_id': 79699


## order_payments
- There are multiple order_id for multiple payments  
For example: if for an order_id, max payment_sequential is 6 then that means there are 6 payments done for that order_id

- order_id, payment_sequential together make primary key
- order_id: id for each order
- payment_sequential: $i_{th}$ payment for given order_id
- payment_type: ['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined']
- payment_installments: tells the number of installments
- payment_value

In [41]:
order_payments = pd.read_csv(r"data\original_data\olist_order_payments_dataset.csv")
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [42]:
order_payments['order_id'].duplicated().sum()

np.int64(4446)

In [50]:
order_payments[order_payments['order_id'].duplicated(keep=False)].sort_values(by='order_id').head(15)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
80856,0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
89575,0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63
20036,002f19a65a2ddd70a090297872e6d64e,1,voucher,1,44.11
98894,002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
30155,0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
10244,0071ee2429bc1efdc43aa3e073a5290e,1,voucher,1,100.00
16459,009ac365164f8e06f59d18a08045f6c4,2,voucher,1,4.50
15298,009ac365164f8e06f59d18a08045f6c4,6,voucher,1,4.17
32058,009ac365164f8e06f59d18a08045f6c4,4,voucher,1,5.45
285,009ac365164f8e06f59d18a08045f6c4,5,voucher,1,8.75


In [75]:
order_payments['payment_type'].value_counts(normalize=True)*100

payment_type
credit_card    73.922376
boleto         19.043952
voucher         5.558978
debit_card      1.471806
not_defined     0.002888
Name: proportion, dtype: float64

## order_reviews_dataset
- One thing i can see is:   
    find the order date of order_id and compare it with review_answer_timestamp and from there take reviews which were given after some time of using the product and analyse those reviews
- For multiple order_id we can have same review_id
- For multiple review_id we can have same order_id
- For primary key use composite - order_id, review_id
- Scores are out of 5


In [139]:
order_reviews = pd.read_csv(r'data\original_data\olist_order_reviews_dataset.csv')
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [141]:
print("Duplicated order_id: ", order_reviews['order_id'].duplicated().sum())
print("Duplicated reivew_id: ", order_reviews['review_id'].duplicated().sum())

print("Duplicated (order_id, review_id):",order_reviews[['order_id','review_id']].duplicated().sum())

Duplicated order_id:  551
Duplicated reivew_id:  814
Duplicated (order_id, review_id): 0


In [107]:
order_reviews['review_score'].value_counts(normalize=True)*100

review_score
5    57.776344
4    19.291704
1    11.513344
3     8.242965
2     3.175643
Name: proportion, dtype: float64

In [108]:
print("%age Null review_comment_title: ",round(order_reviews['review_comment_title'].isnull().sum()/len(order_reviews)*100, 2),"%")
order_reviews[~order_reviews['review_comment_title'].isnull()].head()

%age Null review_comment_title:  88.34 %


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22 00:00:00,2018-05-23 16:45:47
15,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,Super recomendo,"Vendedor confiável, produto ok e entrega antes...",2018-05-23 00:00:00,2018-05-24 03:00:01
19,373cbeecea8286a2b66c97b1b157ec46,583174fbe37d3d5f0d6661be3aad1786,1,Não chegou meu produto,Péssimo,2018-08-15 00:00:00,2018-08-15 04:10:37
22,d21bbc789670eab777d27372ab9094cc,4fc44d78867142c627497b60a7e0228a,5,Ótimo,Loja nota 10,2018-07-10 00:00:00,2018-07-11 14:10:25
34,c92cdd7dd544a01aa35137f901669cdf,37e7875cdce5a9e5b3a692971f370151,4,Muito bom.,Recebi exatamente o que esperava. As demais en...,2018-06-07 00:00:00,2018-06-09 18:44:02


## orders_dataset
- order_id: primary key
- customer_id: Each order has a unique customer_id. (customer_unique_id is the unique identifier of a customer).   
customer_id will be used as foreign key

In [110]:
orders = pd.read_csv(r'data\original_data\olist_orders_dataset.csv')
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [118]:
print("Duplicated order_id: ", orders['order_id'].duplicated().sum())
print("Duplicated customer_id: ",orders['customer_id'].duplicated().sum())

Duplicated order_id:  0
Duplicated customer_id:  0


In [121]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

##  products_dataset
- product_id: primary_key

In [27]:
products = pd.read_csv(r'data\original_data\olist_products_dataset.csv')
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [124]:
products['product_id'].duplicated().sum()

np.int64(0)

## sellers_dataset
- seller_id: primary key
- It has the same issue as customers which is - there are zip_codes in sellers which are not in geolocation from which sellers table refer for zip_codes in database

In [40]:
sellers = pd.read_csv(r'data\original_data\olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix':str})
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [41]:
sellers['seller_id'].duplicated().sum()

np.int64(0)

## product_category_name_translation
- replace product_category_name with its english translation 
- and save the csv file

In [29]:
name_translation = pd.read_csv(r'data\original_data\product_category_name_translation.csv')
print(len(name_translation))
name_translation.head()

71


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [30]:
print(len(products))
products.head()

32951


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [36]:
# Replace Portuguese category names in products with the English translation
translation_map = dict(
    zip(name_translation['product_category_name'], name_translation['product_category_name_english'])
)

products['product_category_name'] = (
    products['product_category_name']
    .map(translation_map)
    .fillna(products['product_category_name'])
)

products = products.astype(dtype={'product_name_lenght':'Int16', 'product_description_lenght':'Int16', 'product_photos_qty':'Int16'})

In [37]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  Int16  
 3   product_description_lenght  32341 non-null  Int16  
 4   product_photos_qty          32341 non-null  Int16  
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: Int16(3), float64(4), str(2)
memory usage: 1.8 MB


In [38]:
## Save to csv
products.to_csv(r'data\transformed_data\olist_products_dataset_transformed.csv', index=False)